# Profile a decode loop, from slow to fixed
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week6/Decode_Loop_Profile_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 6, Module 3: The overhead term**

The roofline covers time when the device is computing or moving data. The overhead term is the rest: time when neither is busy, because the device is waiting for the host. The lecture traces a slow greedy decode loop with three planted problems and fixes them one at a time:

1. The attention mask grows by one column every step. With dynamic shapes off, every step recompiles until the recompile limit, and then the step runs in eager mode.
2. A call to `.item()` checks for the end-of-sequence token inside the compiled step, before the step computes the log-probability of the chosen token for logging. It splits the step into two compiled regions and makes the host wait for the device every step.
3. A new position tensor is built on the host every step and copied to the device.

In this notebook you build the same loop around a small two-layer decoder, so that compiles take seconds, not minutes. You run three versions, profile each, and read the evidence for each problem in the profiler's numbers and in a Chrome trace you can open in Perfetto.

**What you will do**
1. Build a small decoder with a KV cache.
2. Run version 1, with all three problems, and see the recompile storm.
3. Run version 2, with a static cache and a padded mask, and find the graph break and the host copy.
4. Run version 3, with the end-of-sequence check and the position counter moved, and confirm the step is one region.
5. Compare step times and check that all three versions generate the same tokens.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. The notebook takes about three minutes, most of it the compiles in version 1.

## 1. A small decoder
Two transformer layers, model width 512, 8 heads, a 8000-word vocabulary and room for 128 positions. The `forward` method supports both cache styles from the lecture:

- **growing:** each step concatenates the new key and value to the cache, so the cache gets one position longer.
- **static:** the cache is allocated once for 128 positions, and each step writes its key and value into slot `pos` in place.

The mask is added to the attention scores. Padded positions get -1e4, a finite value, so the same code is safe in FP16.

In [ ]:
import json
import statistics
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.profiler import profile, ProfilerActivity, record_function

assert torch.cuda.is_available(), "This notebook needs a GPU: Runtime > Change runtime type > T4 GPU"
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
dev = "cuda"
V, D, H, LAYERS, MAX = 8000, 512, 8, 2, 128
HD = D // H
EOS = 7999                                   # a token id that greedy decoding will not produce here


class Layer(nn.Module):
    def __init__(self):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.qkv, self.o = nn.Linear(D, 3 * D), nn.Linear(D, D)
        self.up, self.down = nn.Linear(D, 4 * D), nn.Linear(4 * D, D)

    def forward(self, x, k_cache, v_cache, mask, pos, static):
        q, k, v = self.qkv(self.n1(x)).view(1, 1, 3, H, HD).unbind(2)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))                 # [1, H, 1, HD]
        if static:
            k_cache.index_copy_(2, pos, k)                                # write slot pos in place
            v_cache.index_copy_(2, pos, v)
        else:
            k_cache = torch.cat([k_cache, k], dim=2)                      # the cache grows by one
            v_cache = torch.cat([v_cache, v], dim=2)
        scores = q @ k_cache.transpose(-2, -1) / HD ** 0.5 + mask
        att = torch.softmax(scores, dim=-1) @ v_cache
        x = x + self.o(att.transpose(1, 2).reshape(1, 1, D))
        return x + self.down(F.gelu(self.up(self.n2(x)))), k_cache, v_cache


class ToyDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb, self.pos_emb = nn.Embedding(V, D), nn.Embedding(MAX, D)
        self.layers = nn.ModuleList(Layer() for _ in range(LAYERS))
        self.head = nn.Linear(D, V)

    def forward(self, tok, caches, mask, pos, static):
        x = self.emb(tok) + self.pos_emb(pos)
        new = []
        for layer, (k, v) in zip(self.layers, caches):
            x, k, v = layer(x, k, v, mask, pos, static)
            new.append((k, v))
        return self.head(x), new


torch.manual_seed(0)
model = ToyDecoder().to(dev).eval()
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f} M parameters")

## 2. Three versions of the decode loop
| | Version 1 | Version 2 | Version 3 |
|---|---|---|---|
| KV cache | grows each step | static, 128 slots | static, 128 slots |
| mask | grows by one column | 128 columns, built on the device | 128 columns, built on the device |
| end-of-sequence check | `.item()` inside the step | `.item()` inside the step | outside the step, every 8 tokens |
| position | `torch.tensor([t])` on the host, copied | `torch.tensor([t])` on the host, copied | a device counter, `pos += 1` |
| compile | `dynamic=False` | `dynamic=False` | `dynamic=False` |

Each version generates 32 tokens greedily from the same start token. `run_loop` records the time of every step, then profiles three steady steps and saves a Chrome trace.

In [ ]:
def empty_caches(static):
    T = MAX if static else 0
    return [(torch.zeros(1, H, T, HD, device=dev), torch.zeros(1, H, T, HD, device=dev)) for _ in range(LAYERS)]


def make_step(version):
    static = version > 1

    def step(tok, caches, mask, pos):
        logits, caches = model(tok, caches, mask, pos, static)
        nxt = logits[:, -1].argmax(-1, keepdim=True)
        if version < 3 and (nxt == EOS).item():    # host check inside the step: a graph break
            pass
        logprob = torch.log_softmax(logits[:, -1].float(), -1).gather(-1, nxt)   # kept for logging
        return nxt, caches, logprob

    return torch.compile(step, dynamic=False)


def run_loop(version, n_tokens=32, profile_steps=3):
    torch._dynamo.reset()
    torch._dynamo.utils.counters.clear()          # count compiles for this version only
    step = make_step(version)
    static = version > 1
    caches = empty_caches(static)
    tok = torch.zeros(1, 1, dtype=torch.long, device=dev)
    pos = torch.zeros(1, dtype=torch.long, device=dev)
    slots = torch.arange(MAX, device=dev)
    tokens, times, prof = [], [], None

    def one_step(t):
        nonlocal tok, caches, pos
        if version < 3:
            pos = torch.tensor([t], device=dev)                    # built on the host, copied each step
        if static:
            mask = torch.where(slots <= pos, 0.0, -1e4).view(1, 1, 1, MAX)
        else:
            mask = torch.zeros(1, 1, 1, t + 1, device=dev)          # one column longer every step
        tok, caches, _ = step(tok, caches, mask, pos)
        if version == 3:
            pos += 1                                                 # stays on the device
            if (t + 1) % 8 == 0 and (tok == EOS).any().item():       # check every 8 tokens
                return False
        tokens.append(tok)
        return True

    with torch.inference_mode():
        for t in range(n_tokens):
            if t == n_tokens - profile_steps:
                prof = profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA])
                prof.__enter__()
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            with record_function("decode_step"):
                one_step(t)
            torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1e3)
        prof.__exit__(None, None, None)
    path = f"/content/trace_version{version}.json"
    prof.export_chrome_trace(path)
    graphs = torch._dynamo.utils.counters["stats"]["unique_graphs"]
    return {"times": times, "tokens": torch.cat(tokens).flatten().tolist(), "prof": prof, "graphs": graphs, "trace": path}


def evidence(r, profile_steps=3):
    """Per-step counts from the profiled steps: the numbers the lecture reads off a trace."""
    ka = r["prof"].key_averages()
    def count(match, device=False):
        return sum(e.count for e in ka if match(e.key)
                   and (e.device_type == torch.autograd.DeviceType.CUDA) == device) / profile_steps
    kernels = count(lambda k: True, device=True)
    regions = count(lambda k: "Compiled" in k and "Region" in k)
    items = count(lambda k: k == "aten::item")
    copies = count(lambda k: "Memcpy HtoD" in k, device=True)
    gpu_us = sum(e.self_device_time_total if hasattr(e, "self_device_time_total") else e.self_cuda_time_total
                 for e in ka if e.device_type == torch.autograd.DeviceType.CPU) / profile_steps
    steady = statistics.median(r["times"][-8:])
    return {"kernels": kernels, "regions": regions, "items": items, "copies": copies,
            "gpu_ms": gpu_us / 1e3, "step_ms": steady}

## 3. Version 1: all three problems

In [ ]:
v1 = run_loop(1)
print("step times (ms):", " ".join(f"{t:.0f}" if t >= 10 else f"{t:.1f}" for t in v1["times"]))
print("distinct compiled graphs:", v1["graphs"])
e1 = evidence(v1)
print(f"steady step: {e1['step_ms']:.2f} ms, {e1['kernels']:.0f} kernels per step, GPU busy {e1['gpu_ms'] / e1['step_ms']:.0%}")

**Reading the output**
- The first steps each take a second or more: every step compiles again, because the mask is one column longer. Dynamo prints a warning when it hits the recompile limit, which is the lecture's Trace 1.
- After the limit, the step runs in eager mode. The times drop, but each step now launches one kernel per operation, the comb of short kernels the lecture describes.
- The GPU is busy for a small fraction of each step.

## 4. Version 2: static cache and padded mask
The first fix from the lecture: a cache allocated for the maximum length and a mask padded to the maximum length, so every step has the same shapes.

In [ ]:
v2 = run_loop(2)
print("step times (ms):", " ".join(f"{t:.0f}" if t >= 10 else f"{t:.1f}" for t in v2["times"]))
print("distinct compiled graphs:", v2["graphs"])
e2 = evidence(v2)
print(f"steady step: {e2['step_ms']:.2f} ms, {e2['regions']:.0f} compiled region(s), {e2['items']:.0f} .item() call(s), "
      f"{e2['copies']:.0f} host-to-device copy(ies) per step, GPU busy {e2['gpu_ms'] / e2['step_ms']:.0%}")

**Reading the output**
- Only the first step compiles. The storm is gone.
- The step now compiles once per region, but it still runs as two compiled regions, because the `.item()` call splits it. The host waits at the `.item()` until the device finishes the first region, then the device sits idle while Python launches the second region, which computes the log-probability. This is the lecture's Trace 2.
- The host-to-device copy is the position tensor built from `torch.tensor([t])` every step.

A second trap hides in the same line. If the check were written `nxt.item() == EOS`, the integer token would be a Python value inside the compiled function, and Dynamo would add a guard on it, such as `token == 2930`, to the second region. Every new token would then recompile that region until the recompile limit. Comparing on the device first, `(nxt == EOS).item()`, gives the host only a `True` or `False`.

## 5. Version 3: one region, no copies
The remaining fixes: the end-of-sequence check moves out of the compiled step and runs every 8 tokens, and the position stays on the device as a counter.

In [ ]:
v3 = run_loop(3)
print("step times (ms):", " ".join(f"{t:.0f}" if t >= 10 else f"{t:.1f}" for t in v3["times"]))
e3 = evidence(v3)
print(f"steady step: {e3['step_ms']:.2f} ms, {e3['regions']:.0f} compiled region(s), {e3['items']:.0f} .item() call(s), "
      f"{e3['copies']:.0f} host-to-device copy(ies) per step, GPU busy {e3['gpu_ms'] / e3['step_ms']:.0%}")

**Reading the output**
- One compiled region per step, no `.item()` in the steady steps, and no host-to-device copy.
- The step is faster than version 2. The remaining time outside the GPU is the cost of calling one compiled region from Python. `mode="reduce-overhead"` would wrap it in a CUDA Graph, the next fix in the lecture's list.

## 6. Compare, and check the tokens
After each fix, benchmark again without the profiler and check that the output still matches. The step times below come from the steps that ran without the profiler.

In [ ]:
print(f"{'version':>7} {'step time':>10} {'kernels':>8} {'regions':>8} {'.item()':>8} {'H2D copies':>11} {'GPU busy':>9}")
for name, e in (("1", e1), ("2", e2), ("3", e3)):
    print(f"{name:>7} {e['step_ms']:7.2f} ms {e['kernels']:8.0f} {e['regions']:8.0f} {e['items']:8.0f} {e['copies']:11.0f} {e['gpu_ms'] / e['step_ms']:9.0%}")

same_12 = v1["tokens"] == v2["tokens"]
same_23 = v2["tokens"] == v3["tokens"]
print(f"\nTokens identical, version 1 and 2: {same_12}; version 2 and 3: {same_23}")
print("first tokens:", v3["tokens"][:12])

**Reading the output**
- Each fix removes one source of waiting, and the step time falls each time.
- Padding the mask to 128 positions means the attention kernels compute scores for positions that do not exist yet. When you place this kernel on a roofline, count only the useful FLOPs, so the padding shows up as a gap.
- The generated tokens are the same in all three versions. A fix that changes the output is not a fix.

## 7. Open a trace in Perfetto
Each version saved a Chrome trace of three steady steps. To read one:

1. Open the folder icon on the left, right-click `trace_version2.json` and choose **Download**. Or run the next cell.
2. Go to https://ui.perfetto.dev and drag the file onto the page.
3. Use **W** and **S** to zoom, **A** and **D** to pan.

Look for the patterns from the lecture: a `Torch-Compiled Region` span on the CPU row for each region, the long `aten::item` with a synchronize inside it in version 2, the gap on the device row right after it, and the small `Memcpy HtoD` before each step. In version 3, each step is one region and the gaps are shorter.

In [ ]:
try:
    from google.colab import files
    files.download("/content/trace_version2.json")
except ImportError:
    print("Not running in Colab. The traces are in /content:", ", ".join(f"trace_version{v}.json" for v in (1, 2, 3)))

## Try this
1. In version 3, compile with `mode="reduce-overhead"` in `make_step`. How much of the remaining step time goes away?
2. In version 2, remove the `.item()` check only. Which number changes, and which stays?
3. Change `LAYERS` to 8 and run version 1. How long does the storm take now?

## Summary
- Time when neither compute nor memory is busy is the overhead term. The profiler shows it as gaps on the device row.
- A growing mask with dynamic shapes off is a recompile storm. A static cache and a padded mask fix it.
- `.item()` inside a compiled step is a graph break and a synchronization every step. Check rarely, outside the step.
- A tensor built on the host every step is a host-to-device copy on the critical path. Keep counters on the device.
- After each fix, benchmark without the profiler and check that the output is unchanged.

Lecture: Week 6, Module 3, slides 29 to 38 (sources of scheduling cost, how the profiler collects events, profiling decode steps, anatomy of a trace, five patterns, the slow decode loop worked example).